# 01.08 - SSD and DETR comparison

**Notebook type:** Solution notebook with theory, complete implementations, smoke checks, and test cases.

**Daily output:** SSD-versus-DETR offline benchmark notebook.

Pretrained SSD and DETR weights normally require a download. To keep this notebook reproducible and within the allowed libraries, deterministic raw outputs are provided at the exact post-processing boundary. You will normalize both detector families to one schema, handle NMS correctly, benchmark adapters, and analyze misses and duplicates. Replace the fixtures with real model outputs later without changing the comparison API.

## Core Ideas

- **SSD** is anchor-based: dense predictions at multiple feature-map scales are decoded, confidence-filtered, and usually passed through class-aware NMS.
- **DETR** predicts a fixed-size set of object queries and uses bipartite matching during training. A `no-object` class absorbs unused queries, and standard DETR does not require NMS.
- SSD commonly returns pixel `xyxy` boxes after model post-processing. Raw DETR outputs are class logits plus normalized `cxcywh` boxes; adapters must remove `no-object`, convert coordinates, and scale to pixels.
- A fair comparison uses the same images, confidence rule, output schema, device synchronization, and evaluation matcher. Adapter latency alone is not end-to-end model latency.
- Confidence scores from different architectures are not automatically calibrated. Similar score thresholds do not guarantee similar precision.

## Allowed-Library Pretrained Route

- **SSD:** Torchvision provides `ssd300_vgg16` with `SSD300_VGG16_Weights.DEFAULT` and `ssdlite320_mobilenet_v3_large` with official weights. `weights=None, weights_backbone=None` constructs the architecture without downloading.
- **DETR:** The allowlisted `transformers` package provides `AutoImageProcessor` and `AutoModelForObjectDetection`; `facebook/detr-resnet-50` is an official pretrained checkpoint route. `local_files_only=True` makes cache requirements explicit and fails rather than silently downloading.
- Before competition use, confirm that external weights are permitted and pre-cache or attach them. The deterministic outputs below keep the adapter/evaluation lesson runnable without network or large checkpoints; their schemas match the real model boundaries.

Official references: https://docs.pytorch.org/vision/stable/models/ssd.html and https://huggingface.co/docs/transformers/model_doc/detr

In [ ]:
import time
import numpy as np
import torch
from torchvision.ops import nms
import pandas as pd

SEED = 1
np.random.seed(SEED)
torch.manual_seed(SEED)
CLASS_NAMES = ["circle", "square"]
IMAGE_SIZES = [(80, 100), (100, 120), (90, 90)]  # (height, width)

## Prepared Canonical Raw Outputs

The three-image batch has six ground-truth objects. SSD fixtures include overlapping anchor-derived duplicates. DETR fixtures include a final `no-object` logit and unused queries. Scores are deliberately imperfect so the shared error analysis has true positives, false positives, and misses.

In [ ]:
GROUND_TRUTH = [
    {"boxes": torch.tensor([[8, 10, 32, 35], [58, 35, 88, 68]], dtype=torch.float32), "labels": torch.tensor([0, 1])},
    {"boxes": torch.tensor([[15, 18, 48, 55], [70, 55, 108, 90]], dtype=torch.float32), "labels": torch.tensor([1, 0])},
    {"boxes": torch.tensor([[12, 12, 38, 42], [48, 45, 78, 78]], dtype=torch.float32), "labels": torch.tensor([0, 1])},
]

SSD_RAW = [
    {"boxes": torch.tensor([[8,10,32,35],[9,11,33,36],[58,35,88,68],[3,55,20,75]], dtype=torch.float32), "scores": torch.tensor([.94,.74,.87,.42]), "labels": torch.tensor([0,0,1,1])},
    {"boxes": torch.tensor([[15,18,48,55],[16,19,49,56],[72,56,109,91]], dtype=torch.float32), "scores": torch.tensor([.91,.63,.48]), "labels": torch.tensor([1,1,0])},
    {"boxes": torch.tensor([[12,12,38,42],[50,46,79,79],[5,60,25,84]], dtype=torch.float32), "scores": torch.tensor([.89,.76,.39]), "labels": torch.tensor([0,1,0])},
]

# DETR logits contain [circle, square, no-object]. Boxes are normalized cxcywh.
DETR_RAW = [
    {"logits": torch.tensor([[4.2,.2,-1.0],[.1,3.7,-.5],[.3,.2,3.6]]), "pred_boxes": torch.tensor([[.20,.28125,.24,.3125],[.73,.64375,.30,.4125],[.50,.50,.20,.20]])},
    {"logits": torch.tensor([[.1,4.0,-.8],[3.0,.3,-.5],[.2,.1,3.4]]), "pred_boxes": torch.tensor([[.2625,.365,.275,.37],[.7417,.725,.3167,.35],[.4,.4,.2,.2]])},
    {"logits": torch.tensor([[3.8,.1,-.6],[.2,2.7,-.3],[2.0,.1,-.2]]), "pred_boxes": torch.tensor([[.2778,.30,.2889,.3333],[.70,.6833,.3333,.3667],[.20,.80,.22,.20]])},
]

print("images:", len(IMAGE_SIZES), "ground-truth objects:", sum(len(item["labels"]) for item in GROUND_TRUTH))

## Exercise 01-A: Normalize SSD output

Filter by confidence, then apply class-aware `torchvision.ops.nms`. Do not let one class suppress another.

**Return structure — `normalize_ssd_output`:** A dictionary with `boxes`: CPU `torch.float32 [K,4]` pixel `xyxy`; `scores`: CPU `torch.float32 [K]`; and `labels`: CPU `torch.int64 [K]`. Rows are ordered by descending score and `0 <= K <= N`.

In [ ]:
def normalize_ssd_output(raw_output, confidence_threshold=0.4, iou_threshold=0.5):
    boxes = raw_output["boxes"].detach().cpu().to(torch.float32)
    scores = raw_output["scores"].detach().cpu().to(torch.float32)
    labels = raw_output["labels"].detach().cpu().to(torch.int64)
    confident = torch.where(scores >= confidence_threshold)[0]
    boxes, scores, labels = boxes[confident], scores[confident], labels[confident]
    kept_parts = []
    for class_id in torch.unique(labels):
        class_indices = torch.where(labels == class_id)[0]
        kept_parts.append(class_indices[nms(boxes[class_indices], scores[class_indices], iou_threshold)])
    if kept_parts:
        kept = torch.cat(kept_parts)
        kept = kept[torch.argsort(scores[kept], descending=True)]
    else:
        kept = torch.empty(0, dtype=torch.int64)
    return {"boxes": boxes[kept], "scores": scores[kept], "labels": labels[kept]}


# Smoke check: normalize and suppress duplicate SSD predictions.
ssd_smoke = normalize_ssd_output(SSD_RAW[0])
print("normalized SSD:", {key: value.shape for key, value in ssd_smoke.items()})
print("SSD scores:", ssd_smoke["scores"].tolist())

## Exercise 01-B: Normalize DETR output

Softmax over all classes, drop queries whose winning class is `no-object`, filter by score, convert normalized `cxcywh` to pixel `xyxy`, and clamp to the image. Do not apply NMS.

**Return structure — `normalize_detr_output`:** A dictionary with `boxes`: CPU `torch.float32 [K,4]` pixel `xyxy`; `scores`: CPU `torch.float32 [K]`; and `labels`: CPU `torch.int64 [K]`. Rows are ordered by descending score and contain no `no-object` labels.

In [ ]:
def normalize_detr_output(raw_output, image_size, confidence_threshold=0.4):
    probabilities = torch.softmax(raw_output["logits"].detach().cpu(), dim=1)
    object_scores, object_labels = probabilities[:, :-1].max(dim=1)
    all_labels = probabilities.argmax(dim=1)
    no_object_id = probabilities.shape[1] - 1
    kept = torch.where((all_labels != no_object_id) & (object_scores >= confidence_threshold))[0]
    boxes = raw_output["pred_boxes"].detach().cpu().to(torch.float32)[kept]
    height, width = image_size
    xc, yc, box_width, box_height = boxes.unbind(dim=1)
    pixel_boxes = torch.stack([
        (xc - box_width / 2.0) * width,
        (yc - box_height / 2.0) * height,
        (xc + box_width / 2.0) * width,
        (yc + box_height / 2.0) * height,
    ], dim=1)
    pixel_boxes[:, 0::2].clamp_(0.0, float(width))
    pixel_boxes[:, 1::2].clamp_(0.0, float(height))
    scores = object_scores[kept].to(torch.float32)
    labels = object_labels[kept].to(torch.int64)
    order = torch.argsort(scores, descending=True)
    return {"boxes": pixel_boxes[order], "scores": scores[order], "labels": labels[order]}


# Smoke check: normalize DETR queries without applying NMS.
detr_smoke = normalize_detr_output(DETR_RAW[0], IMAGE_SIZES[0])
print("normalized DETR:", {key: value.shape for key, value in detr_smoke.items()})
print("DETR boxes:", detr_smoke["boxes"])

## Exercise 01-C: Match predictions to ground truth

Greedily visit predictions by descending confidence. A prediction is a TP only when its class matches an unused truth box and IoU meets the threshold. Other predictions are FP; unmatched truths are FN.

**Return structure — `match_detections`:** A dictionary with integer `tp`, `fp`, and `fn`; `matches`, a list of `(prediction_index, truth_index, iou_float)` tuples; `false_positive_indices`, a list of prediction indices; and `false_negative_indices`, a list of truth indices. Every index refers to the supplied, unmodified inputs.

In [ ]:
def match_detections(prediction, truth, iou_threshold=0.5):
    order = torch.argsort(prediction["scores"], descending=True)
    unused_truth = set(range(len(truth["labels"])))
    matches = []
    false_positive_indices = []
    for prediction_index_tensor in order:
        prediction_index = int(prediction_index_tensor)
        box = prediction["boxes"][prediction_index]
        label = int(prediction["labels"][prediction_index])
        best_truth = -1
        best_iou = -1.0
        for truth_index in sorted(unused_truth):
            if int(truth["labels"][truth_index]) != label:
                continue
            truth_box = truth["boxes"][truth_index]
            top_left = torch.maximum(box[:2], truth_box[:2])
            bottom_right = torch.minimum(box[2:], truth_box[2:])
            intersection_wh = (bottom_right - top_left).clamp(min=0)
            intersection = float(intersection_wh[0] * intersection_wh[1])
            area_box = float((box[2] - box[0]).clamp(min=0) * (box[3] - box[1]).clamp(min=0))
            area_truth = float((truth_box[2] - truth_box[0]).clamp(min=0) * (truth_box[3] - truth_box[1]).clamp(min=0))
            iou = intersection / max(area_box + area_truth - intersection, 1e-7)
            if iou > best_iou:
                best_iou = iou
                best_truth = truth_index
        if best_truth >= 0 and best_iou >= iou_threshold:
            matches.append((prediction_index, best_truth, float(best_iou)))
            unused_truth.remove(best_truth)
        else:
            false_positive_indices.append(prediction_index)
    false_negative_indices = sorted(unused_truth)
    return {
        "tp": len(matches),
        "fp": len(false_positive_indices),
        "fn": len(false_negative_indices),
        "matches": matches,
        "false_positive_indices": false_positive_indices,
        "false_negative_indices": false_negative_indices,
    }


# Smoke check: match the first SSD image against the shared truth.
ssd_match_smoke = match_detections(ssd_smoke, GROUND_TRUTH[0])
print("SSD image-0 match:", ssd_match_smoke)

## Exercise 01-D: Benchmark both adapters

Run each adapter repeatedly over the complete prepared batch. This measures only output normalization, not neural-network inference. Use `time.perf_counter` and report milliseconds per image.

**Return structure — `benchmark_adapters`:** A `pandas.DataFrame` with exactly two rows named `SSD` and `DETR` in column `model`, plus numeric columns `images`, `repeats`, `kept_predictions`, and `adapter_ms_per_image`. Counts summarize all repeated calls; latency is a non-negative Python-compatible float.

In [ ]:
def benchmark_adapters(ssd_outputs, detr_outputs, image_sizes, repeats=200):
    records = []
    start = time.perf_counter()
    ssd_kept = 0
    for _ in range(repeats):
        for raw_output in ssd_outputs:
            ssd_kept += len(normalize_ssd_output(raw_output)["scores"])
    elapsed = time.perf_counter() - start
    records.append({
        "model": "SSD", "images": len(ssd_outputs), "repeats": repeats,
        "kept_predictions": ssd_kept,
        "adapter_ms_per_image": 1000.0 * elapsed / (repeats * len(ssd_outputs)),
    })
    start = time.perf_counter()
    detr_kept = 0
    for _ in range(repeats):
        for raw_output, image_size in zip(detr_outputs, image_sizes):
            detr_kept += len(normalize_detr_output(raw_output, image_size)["scores"])
    elapsed = time.perf_counter() - start
    records.append({
        "model": "DETR", "images": len(detr_outputs), "repeats": repeats,
        "kept_predictions": detr_kept,
        "adapter_ms_per_image": 1000.0 * elapsed / (repeats * len(detr_outputs)),
    })
    return pd.DataFrame(records, columns=["model", "images", "repeats", "kept_predictions", "adapter_ms_per_image"])


# Smoke check: benchmark both adapters on the complete prepared batch.
adapter_benchmark = benchmark_adapters(SSD_RAW, DETR_RAW, IMAGE_SIZES)
print(adapter_benchmark.to_string(index=False))

## Exercise 01-E: Produce a shared error report

Normalize both model families and match them against the same complete ground-truth batch. Aggregate TP, FP, FN, precision, recall, and F1. Also preserve image-level details for inspection.

**Return structure — `compare_detectors`:** A `pandas.DataFrame` with two rows (`SSD`, `DETR`) and columns `model`, `images`, `truth_objects`, `predictions`, `tp`, `fp`, `fn`, `precision`, `recall`, and `f1`; plus a `list[dict]` of length six containing one detail per model-image pair. Each detail has `model` (`str`), `image_index` (`int`), `tp`, `fp`, `fn` (`int`), `matches` (`list[tuple]`), `false_positive_indices` (`list[int]`), and `false_negative_indices` (`list[int]`).

In [ ]:
def compare_detectors(ssd_outputs, detr_outputs, truths, image_sizes):
    summary_rows = []
    details = []
    families = [
        ("SSD", [normalize_ssd_output(output) for output in ssd_outputs]),
        ("DETR", [normalize_detr_output(output, size) for output, size in zip(detr_outputs, image_sizes)]),
    ]
    for model_name, predictions in families:
        total_tp = total_fp = total_fn = total_predictions = 0
        for image_index, (prediction, truth) in enumerate(zip(predictions, truths)):
            result = match_detections(prediction, truth)
            total_tp += result["tp"]
            total_fp += result["fp"]
            total_fn += result["fn"]
            total_predictions += len(prediction["scores"])
            details.append({"model": model_name, "image_index": image_index, **result})
        precision = total_tp / max(total_tp + total_fp, 1)
        recall = total_tp / max(total_tp + total_fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-12)
        summary_rows.append({
            "model": model_name,
            "images": len(truths),
            "truth_objects": sum(len(truth["labels"]) for truth in truths),
            "predictions": total_predictions,
            "tp": total_tp,
            "fp": total_fp,
            "fn": total_fn,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })
    columns = ["model", "images", "truth_objects", "predictions", "tp", "fp", "fn", "precision", "recall", "f1"]
    return pd.DataFrame(summary_rows, columns=columns), details


# Smoke check: compare both families over every prepared image.
comparison_table, error_details = compare_detectors(SSD_RAW, DETR_RAW, GROUND_TRUTH, IMAGE_SIZES)
print(comparison_table.to_string(index=False))
print("detail rows:", len(error_details))

## Test Cases

Tests focus on schema, coordinate conversion, accounting, and full-batch coverage. They do not assert that one detector family must win.

**Return structure — `run_day01_tests`:** Returns `None`. Success is communicated by assertions completing and the exact printed message `Day 01 tests passed`.

In [ ]:
def run_day01_tests():
    for normalized in [normalize_ssd_output(SSD_RAW[0]), normalize_detr_output(DETR_RAW[0], IMAGE_SIZES[0])]:
        assert set(normalized) == {"boxes", "scores", "labels"}
        count = len(normalized["scores"])
        assert normalized["boxes"].shape == (count, 4)
        assert normalized["boxes"].dtype == torch.float32
        assert normalized["scores"].dtype == torch.float32
        assert normalized["labels"].dtype == torch.int64
        assert torch.all(normalized["scores"][:-1] >= normalized["scores"][1:])
    assert len(ssd_smoke["scores"]) == 3, "NMS should remove the duplicate class-0 SSD box"
    assert torch.allclose(detr_smoke["boxes"][0], torch.tensor([8.0, 10.0, 32.0, 35.0]), atol=0.1)
    match = match_detections(ssd_smoke, GROUND_TRUTH[0])
    assert match["tp"] + match["fp"] == len(ssd_smoke["scores"])
    assert match["tp"] + match["fn"] == len(GROUND_TRUTH[0]["labels"])
    assert adapter_benchmark["model"].tolist() == ["SSD", "DETR"]
    assert (adapter_benchmark["adapter_ms_per_image"] >= 0).all()
    assert comparison_table["images"].tolist() == [3, 3]
    assert comparison_table["truth_objects"].tolist() == [6, 6]
    assert len(error_details) == 6
    for _, row in comparison_table.iterrows():
        assert row["tp"] + row["fp"] == row["predictions"]
        assert row["tp"] + row["fn"] == row["truth_objects"]
    print("Day 01 tests passed")


run_day01_tests()

## Day 01 Checklist

- [ ] I can contrast SSD's anchors and NMS with DETR's object queries and set prediction.
- [ ] I normalized both detector families to the same boxes/scores/labels schema.
- [ ] I removed DETR's no-object queries and avoided unnecessary NMS.
- [ ] I compared every prepared image with a common matching rule.
- [ ] I separated adapter timing from end-to-end inference latency.
- [ ] I can swap real cached model outputs into these adapter APIs later.